# 05 - Multi-Turn Conversations

Every `Runner.run_sync` call you've made so far has been independent -- the agent has no memory of previous calls. That's a problem for a chat bot: if someone tells it their name and then asks "what's my name?", it won't know.


In [ ]:
%pip install -q openai-agents

from agents import set_default_openai_key

# Don't share or commit this notebook with your key filled in.
OPENAI_API_KEY = "sk-..."  # <-- paste your key here
set_default_openai_key(OPENAI_API_KEY)

print("Ready to go.")

## Watching the memory loss


In [ ]:
from agents import Agent, Runner

agent = Agent(name="Assistant", instructions="You are a helpful assistant.")

Runner.run_sync(agent, "My name is Alex.")
result = Runner.run_sync(agent, "What's my name?")
print(result.final_output)

It has no idea -- each call started fresh.

## Carrying context forward

Every result has a `.to_input_list()` method, which returns the full conversation so far in a format you can feed into the next call. Append the next user message to it, and pass the whole list back in.


In [ ]:
result1 = Runner.run_sync(agent, "My name is Alex.")

history = result1.to_input_list()
history.append({"role": "user", "content": "What's my name?"})

result2 = Runner.run_sync(agent, history)
print(result2.final_output)

## A simple chat loop

This is the same pattern `local_chat.py` in your capstone repo uses to let you test your agent in a terminal. Run the cell below and have a real back-and-forth conversation (type `quit` to stop).


In [ ]:
history = []
print("Chatting with the agent. Type 'quit' to stop.\n")

while True:
    user_text = input("You: ").strip()
    if user_text.lower() in {"quit", "exit"}:
        break
    if not user_text:
        continue

    history.append({"role": "user", "content": user_text})
    result = Runner.run_sync(agent, history)
    print(f"Agent: {result.final_output}\n")
    history = result.to_input_list()

## Beyond this course

Manually passing `history` around works well for a notebook or a single terminal session, but for a Telegram bot, messages come in as separate, independent webhook requests -- there's no variable that stays alive between them. Your capstone bot keeps each Telegram message stateless for exactly that reason. If you want to add real cross-message memory to it later, the SDK has a [Sessions](https://openai.github.io/openai-agents-python/sessions/) feature built for storing conversation history somewhere persistent (like a small database) -- worth exploring as a stretch goal, not something to worry about during the course.

**Next:** open `07_from_notebook_to_telegram_bot.ipynb`.
